In [0]:
# ── Step 1: Create database ──────────────────────────────────────
spark.sql("CREATE DATABASE IF NOT EXISTS fraud_db")
spark.sql("USE fraud_db")

# ── Step 2: Define your volume path ─────────────────────────────
# Update catalog and schema names to match yours exactly
CATALOG  = "workspace"
SCHEMA   = "personalproject(pp)"
VOLUME   = "fraud_volume"
FILENAME = "creditcard.csv"

file_path = f"/Volumes/workspace/personalproject(pp)/fraud_volume/creditcard.csv"
print(f"Reading from: {file_path}")

# ── Step 3: Read CSV ─────────────────────────────────────────────
df_raw = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)

# ── Step 4: Preview ──────────────────────────────────────────────
print(f"Total rows    : {df_raw.count()}")
print(f"Total columns : {len(df_raw.columns)}")
print(f"Fraud cases   : {df_raw.filter('Class = 1').count()}")
print(f"Legit cases   : {df_raw.filter('Class = 0').count()}")
df_raw.show(5)

# ── Step 5: Add metadata & write Bronze Delta table ──────────────
from pyspark.sql.functions import current_timestamp, lit

df_bronze = df_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", lit(FILENAME)) \
    .withColumn("pipeline_version", lit("v1.0"))

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fraud_db.bronze_transactions")

print("✅ Bronze table created: fraud_db.bronze_transactions")
print(f"   Total rows written: {df_bronze.count()}")


Reading from: /Volumes/workspace/personalproject(pp)/fraud_volume/creditcard.csv
Total rows    : 284807
Total columns : 31
Fraud cases   : 492
Legit cases   : 284315
+----+------------------+-------------------+----------------+------------------+-------------------+-------------------+-------------------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+--------------------+-------------------+------------------+------------------+------------------+------------------+--------------------+-------------------+------+-----+
|Time|                V1|                 V2|              V3|                V4|                 V5|                 V6|                 V7|                V8|                V9|                V10|               V11|               V12|               V13|               

In [0]:
# Verify Bronze table
df_verify = spark.table("fraud_db.bronze_transactions")

print("=== BRONZE TABLE VERIFIED ===")
print(f"Row count : {df_verify.count()}")
print(f"Columns   : {df_verify.columns}")
df_verify.show(3)

# Check Delta history (time travel is working)
spark.sql(
    "DESCRIBE HISTORY fraud_db.bronze_transactions"
).show(truncate=False)

=== BRONZE TABLE VERIFIED ===
Row count : 284807
Columns   : ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class', 'ingestion_timestamp', 'source_file', 'pipeline_version']
+--------+-------------------+------------------+------------------+-----------------+------------------+-----------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+-----------------+--------------------+-------------------+--------------------+------------------+-----------------+------------------+------------------+------------------+-------------------+-------------------+------+-----+--------------------+--------------+----------------+
|    Time|                 V1|                V2|  

In [0]:
# See all catalogs
spark.sql("SHOW CATALOGS").show()

# See all schemas in your catalog
spark.sql("SHOW SCHEMAS IN workspace").show()

# See all volumes in your schema
spark.sql(
    "SHOW VOLUMES IN workspace.`personalproject(pp)`"
).show()

+---------+
|  catalog|
+---------+
|     main|
|  samples|
|   system|
|workspace|
+---------+

+-------------------+
|       databaseName|
+-------------------+
|            default|
|           fraud_db|
| information_schema|
|personalproject(pp)|
|           projects|
|            salesdb|
+-------------------+

+-------------------+------------+
|           database| volume_name|
+-------------------+------------+
|personalproject(pp)|fraud_volume|
+-------------------+------------+

